# A custom encrypted model from the runtime's operations

A model under CKKS is a sequence of operations on ciphertexts. This notebook writes a small one directly on the
eager API of the runtime (`perseus._pc.Session(...).fhe`): every call runs on the GPU at once, with no recording and no
plan. Its depth fits the chain, so no bootstrap is needed. The four shipped models do the same with `perseus.impl`
(`python/perseus/examples/`), recorded once and replayed under a plan that places their bootstraps.

The model: a feature-wise affine map, a square activation, and a sum over the features,
`y = sum_i (w_i x_i + b_i)^2`, on `n` features held in the first `n` slots of one ciphertext.

In [ ]:
import numpy as np
from perseus import _pc

n = 16                                          # features (slots 0 .. n-1)
steps = [1 << k for k in range(n.bit_length() - 1)]  # rotations of the slot sum: 1, 2, 4, 8

cfg = _pc.Config.preset("n32_gpt2_decode")      # the 32-bit chain the shipped models run on
keys = _pc.KeySource.generate(steps, seed=0)    # secret key + the rotation keys the model uses
sess = _pc.Session(cfg, keys, pool=32, pt_slots=8).complete_setup()
fhe = sess.fhe
print(f"{fhe.slots} slots; setup {sess.setup_seconds()[0]:.1f} s")

## The model in the clear, and encrypted

`encrypt(values, level)` places the values in the slots at a level of the chain (0 = the top: every level
available). `mult_pt` / `add_pt` take plaintext vectors, `square` multiplies a ciphertext by itself (relinearized),
`rotate_and_sum(ct, 1, n)` adds the slots `i .. i + n - 1` into slot `i` with log2(n) rotations.

In [ ]:
rng = np.random.default_rng(0)
S = fhe.slots
x, w, b = (np.zeros(S) for _ in range(3))
x[:n], w[:n], b[:n] = rng.standard_normal(n), rng.standard_normal(n) / np.sqrt(n), 0.1 * rng.standard_normal(n)

y_ref = float(np.sum((w[:n] * x[:n] + b[:n]) ** 2))

ct = fhe.encrypt(x, 0)
h = fhe.add_pt(fhe.mult_pt(ct, w), b)           # w * x + b, feature-wise
h = fhe.square(h)                               # the activation
y = fhe.rotate_and_sum(h, 1, n)                 # sum over the n features, into slot 0
y_fhe = float(np.real(fhe.decrypt_slots_complex(y)[0]))
print(f"encrypted {y_fhe:.10f}  clear {y_ref:.10f}  |err| {abs(y_fhe - y_ref):.2e}; level {ct.level} -> {y.level}")
assert abs(y_fhe - y_ref) < 1e-6 * max(1.0, abs(y_ref))
sess.close()

## Where to go from here

- A deeper model runs out of levels: the shipped models record their forward pass once on the CPU twin
  (`python -m perseus.examples.gpt2.run_decode --fake --capture graphs/<name>`), the planner places the bootstraps
  (`python/perseus/examples/gpt2/make_plan.sh`), and the planned run compiles each block into a program the runtime
  replays (`perseus.impl.driver.PlannedRun`).
- `python/perseus/examples/rmsnorm_from_primitives.py` builds RMSNorm from these primitives with a Chebyshev
  inverse square root; `client_server.ipynb` splits the key holder from the server.